# Import libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (Dense, 
                                    Input, 
                                    LSTM, 
                                    Dropout, 
                                    Conv1D, 
                                    MaxPooling1D, 
                                    Conv2D,
                                    Reshape, 
                                    MaxPooling2D, 
                                    TimeDistributed, 
                                    GlobalAveragePooling1D, 
                                    Concatenate,
                                    Flatten
                                )
from tensorflow.keras.optimizers import (Adam, 
                                         AdamW)
from tensorflow.keras.utils import to_categorical
from sklearn.metrics import classification_report

/Users/kavisanthoshkumar/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [2]:
print("TensorFlow version:", tf.__version__)
print("Available physical devices:")
print(tf.config.list_physical_devices())

print("\nIs MPS available?")
print(tf.config.list_physical_devices('GPU'))

TensorFlow version: 2.18.0
Available physical devices:
[PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU'), PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

Is MPS available?
[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [20]:
data = pd.read_pickle("../data/processed/eeg_bandpass_ica.pkl")
data = data[data["shape"]==(64, 656)]

#### Step 1 : Setting Data in Appropriate Shape

In [ ]:
X = np.stack(data["epoch_data_ICA"].values, axis = 0) # epoch_data_ICA
# Transpose to match (num_trials, n_samples, n_channels)
X = np.transpose(X, (0, 2, 1))

y = data["label"].values

print(f"Shape of X: {X.shape}")
print(f"Shape of y: {y.shape}")

Shape of X: (4083, 656, 64)
Shape of y: (4083,)


#### Step -2 : Normalize Data

In [23]:
# Global Normalization
X = (X-np.mean(X))/np.std(X)

In [24]:
X.shape[0]

4083

#### Step 3: One-hot Encode labels (for classification)

In [25]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
le.fit(y)

# Transform 
y_le = le.transform(y)

# Applying to_categorical 
y_le = to_categorical(y_le, num_classes= len(np.unique(y_le)))
print(f"Shape of y : {y_le.shape}")

Shape of y : (4083, 2)


#### Step 4. Creating tensorflow Dataset

In [26]:
data = tf.data.Dataset.from_tensor_slices((X, y_le))

#### Step 5: Splitting the Tensorflow Dataset into train, test and valid

In [27]:
train_split_ratio = 0.7
val_split_ratio = 0.15
test_split_ratio = 0.15

# Size of each split
train_size = int(train_split_ratio * X.shape[0])
val_size = int(val_split_ratio * X.shape[0])
test_size = int(test_split_ratio * X.shape[0])

# Shuffle the dataset first for a random split
data = data.shuffle(buffer_size= X.shape[0])

training_dataset = data.take(train_size)
val_dataset = data.take(val_size)
test_dataset = data.take(test_size)

print(f"Train dataset size: {len(list(training_dataset.as_numpy_iterator()))}")
print(f"Val dataset size: {len(list(val_dataset.as_numpy_iterator()))}")
print(f"Test dataset size: {len(list(test_dataset.as_numpy_iterator()))}")

Train dataset size: 2858
Val dataset size: 612
Test dataset size: 612


In [28]:
training_dataset = training_dataset.batch(32)
val_dataset = val_dataset.batch(32)
test_dataset = test_dataset.batch(32)

#### Step 6 : CNN MODEL ARCHITECTURE - USING MODEL SUBCLASSING

In [29]:
class EEGCNNLSTM(Model):
    def __init__(self, num_classes):
        super(EEGCNNLSTM, self).__init__()

        # --- Reshape ---
        self.reshape_layer = Reshape((64, 656, 1))        

        # --- CNN feature extractor ---
        self.conv1 = Conv2D(32, (3, 3), activation='leaky_relu', padding='same')
        self.pool1 = MaxPooling2D((2, 2))

        self.conv2 = Conv2D(64, (5, 5), activation='leaky_relu', padding='same')
        self.pool2 = MaxPooling2D((2, 2))

        self.conv3 = Conv2D(128, (7, 7), activation='leaky_relu', padding='same')
        self.pool3 = MaxPooling2D((2, 2))

        self.dropout = Dropout(0.5)

        # Apply TimeDistributed 
        self.flatten = TimeDistributed(Flatten())
        self.fc_time = TimeDistributed(Dense(128, activation='leaky_relu'))
        self.fc_time_1 = TimeDistributed(Dense(128, activation='leaky_relu'))

        # --- Temporal modeling ---
        self.lstm = LSTM(64, return_sequences=True)
        self.global_avg = GlobalAveragePooling1D()

        # --- Final layers ---
        self.concat = Concatenate()
        self.fc_final = Dense(64, activation='leaky_relu')
        self.out_layer = Dense(num_classes, activation='sigmoid')

    def call(self, inputs, training=False):

        # Reshape the input to accomdate the Convolution layers
        x = self.reshape_layer(inputs)

        # CNN feature extraction
        x = self.conv1(x)
        x = self.pool1(x)

        x = self.conv2(x)
        x = self.pool2(x)

        x = self.conv3(x)
        x = self.pool3(x)

        if training:
            x = self.dropout(x)

        x = self.flatten(x)
        x = self.fc_time(x)
        x = self.fc_time_1(x)

        # Expand for temporal modeling
        #x = tf.expand_dims(x, axis=1)  # (batch, time=1, features)
        lstm_out = self.lstm(x)

        # Global Average Pooling
        gap_out = self.global_avg(inputs)

        # Concatenate (to mimic diagram)
        concat_out = self.concat([gap_out, tf.reduce_mean(lstm_out, axis=1)])

        # Final classification
        dense_out = self.fc_final(concat_out)
        output = self.out_layer(dense_out)

        return output


# === Example usage ===
n_samples = 656     # number of time samples
n_channels = 64     # number of EEG electrodes
num_classes = 2     # binary classification

# Instantiate model
model = EEGCNNLSTM(num_classes)

# Build model (needed to show summary)
model.build(input_shape=(None, n_samples, n_channels, 1))
model.summary()


/Users/kavisanthoshkumar/Library/Python/3.9/lib/python/site-packages/keras/src/layers/layer.py:393: UserWarning: `build()` was called on layer 'eegcnnlstm_1', however the layer does not have a `build()` method implemented and it looks like it has unbuilt state. This will cause the layer to be marked as built, despite not being actually built, which may cause failures down the line. Make sure to implement a proper `build()` method.
  warnings.warn(


Model: "eegcnnlstm_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ reshape_1 (Reshape)             │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_5 (MaxPooling2D)  │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_3              │ ?                      │   0 (unbuilt) │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_4              │ ?                      │   0 (unbuilt) │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_5              │ ?                      │   0 (unbuilt) │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d_1      │ ?                      │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ concatenate_1 (Concatenate)     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [30]:
print(model.summary())

f1_metric = tf.keras.metrics.F1Score(average='macro')

# Compile the Model
model.compile(optimizer= Adam(), 
                        loss = "categorical_crossentropy", 
                        metrics = ["accuracy", f1_metric])

Model: "eegcnnlstm_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ reshape_1 (Reshape)             │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_5 (MaxPooling2D)  │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_3              │ ?                      │   0 (unbuilt) │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_4              │ ?                      │   0 (unbuilt) │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_5              │ ?                      │   0 (unbuilt) │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d_1      │ ?                      │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ concatenate_1 (Concatenate)     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

None


In [31]:
history = model.fit(
    training_dataset,
    epochs = 40, 
    batch_size = 32, 
    validation_data = val_dataset
)

Epoch 1/40
90/90 ━━━━━━━━━━━━━━━━━━━━ 20s 165ms/step - accuracy: 0.5211 - f1_score: 0.5184 - loss: 0.7225 - val_accuracy: 0.5049 - val_f1_score: 0.5047 - val_loss: 0.6985
Epoch 2/40
90/90 ━━━━━━━━━━━━━━━━━━━━ 13s 146ms/step - accuracy: 0.5175 - f1_score: 0.5138 - loss: 0.6958 - val_accuracy: 0.5882 - val_f1_score: 0.5619 - val_loss: 0.6826
Epoch 3/40
90/90 ━━━━━━━━━━━━━━━━━━━━ 13s 146ms/step - accuracy: 0.5292 - f1_score: 0.5105 - loss: 0.6876 - val_accuracy: 0.5915 - val_f1_score: 0.5794 - val_loss: 0.6723
Epoch 4/40
90/90 ━━━━━━━━━━━━━━━━━━━━ 13s 146ms/step - accuracy: 0.5901 - f1_score: 0.5748 - loss: 0.6756 - val_accuracy: 0.5915 - val_f1_score: 0.5902 - val_loss: 0.6627
Epoch 5/40
90/90 ━━━━━━━━━━━━━━━━━━━━ 13s 146ms/step - accuracy: 0.6036 - f1_score: 0.5953 - loss: 0.6694 - val_accuracy: 0.5735 - val_f1_score: 0.5647 - val_loss: 0.6750
Epoch 6/40
90/90 ━━━━━━━━━━━━━━━━━━━━ 13s 146ms/step - accuracy: 0.5946 - f1_score: 0.5936 - loss: 0.6665 - val_accuracy: 0.5278 - val_f1_score: 

In [32]:
# Test Predictions
y_val_pred = model.predict(val_dataset)
y_val_pred = np.argmax(y_val_pred, axis = 1)

# Original Predictions
y_val_original = np.concatenate([y.numpy() for x, y in val_dataset])
y_val_original = np.argmax(y_val_original, axis = 1)

20/20 ━━━━━━━━━━━━━━━━━━━━ 2s 59ms/step


2025-10-09 14:42:24.594418: I tensorflow/core/framework/local_rendezvous.cc:405] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


In [33]:
print(classification_report(y_val_original, y_val_pred))

              precision    recall  f1-score   support

           0       0.44      0.26      0.32       300
           1       0.49      0.69      0.57       312

    accuracy                           0.48       612
   macro avg       0.47      0.47      0.45       612
weighted avg       0.47      0.48      0.45       612



# Test Predictions

In [23]:
# Test Predictions
y_test_pred = model.predict(test_dataset)
y_test_pred = np.argmax(y_test_pred, axis = 1)

# Original Predictions
y_test_original = np.concatenate([y.numpy() for x, y in test_dataset])
y_test_original = np.argmax(y_test_original, axis = 1)

20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step


In [24]:
print(classification_report(y_test_original, y_test_pred))

              precision    recall  f1-score   support

           0       0.48      0.42      0.45       289
           1       0.53      0.59      0.56       323

    accuracy                           0.51       612
   macro avg       0.51      0.51      0.50       612
weighted avg       0.51      0.51      0.51       612

